# TensorFlow: Simple quadratic model

* Using TensorFlow.Module
* Using TensorFlow.Data.Dataset
* Using TensorFlow.GradientTape

In [ ]:
import os
from matplotlib import pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

In [ ]:
import tensorflow as tf

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

In [ ]:
# Define a function of dependent variable y = f(x)
def f(x):
  y = x**2 + 2*x - 5
  return y

In [ ]:
def plot_preds(x, y, f, model, title):
  plt.figure()
  plt.plot(x, y, ".", label="Data")
  plt.plot(x, f(x), label="Ground truth")
  plt.plot(x, model(x), label="Predictions")
  plt.title(title)
  plt.legend()

## Generate training data

In [ ]:
# Define x values
x = tf.linspace(-2.0, 2.0, 201)
# Define y values as a function f(x) + some random values
y = f(x) + tf.random.normal(shape=[201])

## Create a model

In [ ]:
#
# Create  a quadratic model with randomly initialized weights and a bias
#
class Model(tf.Module):
  def __init__(self):
    # Randomly generate weight and bias terms
    super().__init__()
    rand_init = tf.random.uniform(shape=[3], minval=0., maxval=5., seed=22)

    # Initialize model parameters
    self.w_q = tf.Variable(initial_value=rand_init[0], trainable=True)
    self.w_l = tf.Variable(initial_value=rand_init[1], trainable=True)
    self.b = tf.Variable(rand_init[2])

  @tf.function
  def __call__(self, x):
    # Quadratic Model : quadratic_weight * x^2 + linear_weight * x + bias
    return tf.multiply(self.w_q, (x**2)) + tf.multiply(self.w_l, x) + self.b

In [ ]:
quad_model = Model()

### Evaluate model before training

In [ ]:
plot_preds(x, y, f, quad_model, "Before training")

## Train a model

Train a model using tf.GradientTape API which use automatic differentiation engine to calculate gradients in order to do trainable variables updating.

In [ ]:
# Define loss function as MSE (Mean Square Error)
def mse_loss(y_pred, y):
  return tf.reduce_mean(tf.square(y_pred - y))

In [ ]:
batch_size = 32

# Create a dataset from training data
dataset = tf.data.Dataset.from_tensor_slices((x, y))
dataset = dataset.shuffle(buffer_size=x.shape[0]).batch(batch_size)

# Set training parameters
epochs = 100
learning_rate = 0.01
losses = []

# Format training loop
for epoch in range(epochs):
  # Iterate over all examples (batches)
  for x_true, y_true in dataset:
    with tf.GradientTape() as tape:
      # Predict y value (getting logits as a raw prediction values)
      logits = quad_model(x_true)
      # Calculate the loss value
      loss_value = mse_loss(y_true, logits)
    # Update parameters w.r.t. gradients
    grads = tape.gradient(loss_value, quad_model.trainable_variables)
    # Apply gradients in order to update trainable variables
    for g,v in zip(grads, quad_model.variables):
        v.assign_sub(learning_rate*g)

  # Keep track of model loss per epoch
  loss = mse_loss(quad_model(x), y)
  losses.append(loss)
  if epoch % 10 == 0:
    print(f"Mean squared error for step {epoch}: {loss.numpy():0.3f}")

# Plot model results
print("\n")
plt.plot(range(epochs), losses)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("MSE loss vs training iterations");

### Evaluate model after training

In [ ]:
plot_preds(x, y, f, quad_model, "After training")